# 02. Agent Memory and Recall

Prerequisite: run `01_ingest_and_deduplicate.ipynb` first so this notebook can reuse the populated FalkorDB graph.

Sample data note:

- The tutorial texts in `notebooks/experimental_data/` are Medium.com articles by Filip Wojcik.
- Source profile: `https://medium.com/@filip.igor.wojcik`
- They are fully accessible without a subscription.

Install prerequisites before running the notebook:

```bash
uv sync --group falkordb --extra notebooks
```


In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import autoroot  # noqa
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext
from rich import print

from grawiki.db import FalkorGraphDB
from grawiki.rag import GraphRAG

load_dotenv()


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
MODEL = (
    os.getenv("GRAWIKI_MODEL")
    or os.getenv("LLM_MODEL")
    or "openrouter:openai/gpt-4.1-mini"
)
EMBEDDING_MODEL = (
    os.getenv("GRAWIKI_EMBEDDING_MODEL")
    or os.getenv("EMBEDDING_MODEL")
    or "openrouter:openai/text-embedding-3-small"
)

database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")
rag = GraphRAG(model=MODEL, embedding_model=EMBEDDING_MODEL, db=database)


@dataclass
class NotebookDeps:
    rag: GraphRAG


def node_payload(node) -> dict[str, Any]:
    payload = node.model_dump()
    payload["labels"] = sorted(node.labels)
    return payload


def hit_payload(hit) -> dict[str, Any]:
    return {
        "score": round(hit.score, 4),
        "matched_on": hit.matched_on,
        "node": node_payload(hit.node),
    }


deps = NotebookDeps(rag=rag)
USER_ID = "demo-user"

In [2]:
agent = Agent(
    model=MODEL,
    deps_type=NotebookDeps,
    system_prompt=(
        "You are a notebook demo agent for GraWiki. Use search_knowledge_graph for facts from the graph, "
        "remember_user_fact when the user tells you something durable about themselves, and "
        "recall_user_memories before answering user-specific follow-up questions."
    ),
)


@agent.tool
async def search_knowledge_graph(
    ctx: RunContext[NotebookDeps],
    query: str,
    limit: int = 5,
) -> list[dict[str, Any]]:
    hits = await ctx.deps.rag.search(query, limit=limit)
    return [hit_payload(hit) for hit in hits]


@agent.tool
async def remember_user_fact(
    ctx: RunContext[NotebookDeps],
    memory_text: str,
    user_id: str,
    related_node_ids: list[str] | None = None,
) -> dict[str, Any]:
    memory = await ctx.deps.rag.remember(
        memory_text,
        metadata={"user_id": user_id},
        related_node_ids=related_node_ids or [],
    )
    return node_payload(memory)


@agent.tool
async def recall_user_memories(
    ctx: RunContext[NotebookDeps],
    query: str,
    user_id: str,
    hops: int = 1,
    limit: int = 5,
) -> list[dict[str, Any]]:
    hits = await ctx.deps.rag.recall(query, user_id=user_id, hops=hops, limit=limit)
    return [hit_payload(hit) for hit in hits]


print("Agent and tools are ready.")

Agent and tools are ready.

## Demo conversation flow

The first call asks the agent to answer a factual question about the graph built in notebook 1.


In [3]:
factual_result = await agent.run(
    "Using the knowledge graph, summarize the main idea behind connecting LLMs and knowledge graphs.",
    deps=deps,
)
print(factual_result.output)

The main idea behind connecting Large Language Models (LLMs) and knowledge graphs is to combine the strengths of 
both systems to enhance the reliability and structure of knowledge used by LLMs. LLMs generate text based on 
probabilistic reasoning over training data but can sometimes hallucinate or produce counterfactual reasoning. By 
enriching LLMs with knowledge graphs and reasoning over them, it is possible to integrate structured, 
domain-specific knowledge that is captured in knowledge graphs with the language understanding capabilities of 
LLMs.

Knowledge graphs are specialized heterogeneous graphs that describe real-world entities, their interrelations, and 
associated properties in a structured and meaningful way. They organize knowledge relevant to particular domains 
and allow complex connections and schema to be modeled. Incorporating knowledge graphs into LLM contexts can help 
provide more accurate, consistent, and explainable information by leveraging the structured data and inference 
capabilities of knowledge graphs alongside the natural language generation of LLMs.

In [4]:
remember_result = await agent.run(
    """My name is Filip and I work as a researcher and data scientist. 
    I focus mostly on knowledge graphs and their integration with large language models.""",
    deps=deps,
)
print(remember_result.output)

Nice to meet you, Filip! If you have any questions or need information related to knowledge graphs, large language 
models, or their integration, feel free to ask. How can I assist you today?

In [5]:
follow_up_result = await agent.run(
    "What do you know about the user Filip and his interests? Use recall before answering.",
    deps=deps,
)
print(follow_up_result.output)

Filip works as a researcher and data scientist focusing mostly on knowledge graphs and their integration with large
language models. Would you like to know more details or discuss anything specific related to these interests?

## Direct recall inspection

The facade-level recall API returns memory hits with readable graph context stored in `node.properties['recall_context']`.


In [7]:
recall_hits = await rag.recall("Filip researcher", hops=2, limit=5)
for hit in recall_hits:
    payload = hit_payload(hit)
    payload["recall_context"] = hit.node.properties.get("recall_context", "")
    print(payload)

{
    'score': 0.6571,
    'matched_on': 'vector',
    'node': {
        'id': '32d8c92e-15e8-4cfe-84d7-60cdab5341ec',
        'labels': ['__memory__'],
        'semantic_key': '32d8c92e-15e8-4cfe-84d7-60cdab5341ec',
        'name': 'Filip works as a researcher and data scientist focusing mostly on knowledge grap',
        'properties': {
            'recall_context': 'Filip works as a researcher and data scientist focusing mostly on knowledge grap 
-[__mentions__]-> Data Scientist\nFilip works as a researcher and data scientist focusing mostly on knowledge grap 
-[__mentions__]-> Filip\nFilip works as a researcher and data scientist focusing mostly on knowledge grap 
-[__mentions__]-> Large Language Models\nFilip works as a researcher and data scientist focusing mostly on 
knowledge grap -[__mentions__]-> Researcher\nFilip -[works_as]-> Data Scientist'
        },
        'embedding': [],
        'content': 'Filip works as a researcher and data scientist focusing mostly on knowledge graphs and their 
integration with large language models.',
        'creation_date': '2026-04-25T17:12:02.165817',
        'metadata': {'user_id': 'Filip'}
    },
    'recall_context': 'Filip works as a researcher and data scientist focusing mostly on knowledge grap 
-[__mentions__]-> Data Scientist\nFilip works as a researcher and data scientist focusing mostly on knowledge grap 
-[__mentions__]-> Filip\nFilip works as a researcher and data scientist focusing mostly on knowledge grap 
-[__mentions__]-> Large Language Models\nFilip works as a researcher and data scientist focusing mostly on 
knowledge grap -[__mentions__]-> Researcher\nFilip -[works_as]-> Data Scientist'
}

{
    'score': 0.6295,
    'matched_on': 'vector',
    'node': {
        'id': '4a19bf82-7f5e-4a3c-bb55-56a587d7ce77',
        'labels': ['__memory__'],
        'semantic_key': '4a19bf82-7f5e-4a3c-bb55-56a587d7ce77',
        'name': "User's name is Filip. He works as a researcher and data scientist focusing mostl",
        'properties': {'recall_context': 'No connected graph context found.'},
        'embedding': [],
        'content': "User's name is Filip. He works as a researcher and data scientist focusing mostly on knowledge 
graphs and their integration with large language models.",
        'creation_date': '2026-04-25T17:29:02.210892',
        'metadata': {'user_id': 'Filip'}
    },
    'recall_context': 'No connected graph context found.'
}

In [8]:
memory_rows = database.ro_query(
    "MATCH (m:__memory__) OPTIONAL MATCH (m)-[r]->(n) RETURN m.id, m.name, type(r), n.id, n.name ORDER BY m.creation_date, m.id LIMIT 20"
).result_set
print({"memory_rows": memory_rows})

{
    'memory_rows': [
        [
            '32d8c92e-15e8-4cfe-84d7-60cdab5341ec',
            'Filip works as a researcher and data scientist focusing mostly on knowledge grap',
            '__mentions__',
            'b7e259a5-8882-4633-a131-e7f17c12ecd5',
            'Data Scientist'
        ],
        [
            '32d8c92e-15e8-4cfe-84d7-60cdab5341ec',
            'Filip works as a researcher and data scientist focusing mostly on knowledge grap',
            '__mentions__',
            '495a5360-d6ba-4d26-9591-b6abf5d2c2cd',
            'Large Language Models'
        ],
        [
            '32d8c92e-15e8-4cfe-84d7-60cdab5341ec',
            'Filip works as a researcher and data scientist focusing mostly on knowledge grap',
            '__mentions__',
            '6ca2698c-9740-4d80-8dd4-44e648426a17',
            'Researcher'
        ],
        [
            '32d8c92e-15e8-4cfe-84d7-60cdab5341ec',
            'Filip works as a researcher and data scientist focusing mostly on knowledge grap',
            '__mentions__',
            'aa82f390-5349-46f1-83a2-d76b4b463711',
            'Filip'
        ],
        [
            '4a19bf82-7f5e-4a3c-bb55-56a587d7ce77',
            "User's name is Filip. He works as a researcher and data scientist focusing mostl",
            None,
            None,
            None
        ]
    ]
}

In [9]:
# Run this once when you are finished with the notebook session.
database.close()